In [2]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications.inception_v3 import preprocess_input
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support

dataset_dir = '/content/drive/MyDrive/NN/dataset'
video_path = '/content/drive/MyDrive/NN/mercedes.mp4'

img_size = (150, 150)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

test_val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    directory=dataset_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_generator = test_val_datagen.flow_from_directory(
    directory=dataset_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

test_generator = test_val_datagen.flow_from_directory(
    directory=dataset_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False,
    subset='validation'
)

def build_xception(input_shape=(150, 150, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(16, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    skip = layers.Conv2D(32, (1, 1), strides=(2, 2), padding='same')(x)
    skip = layers.BatchNormalization()(skip)

    x = layers.SeparableConv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2), strides=(2, 2), padding='same')(x)
    x = layers.add([x, skip])

    skip = layers.Conv2D(64, (1, 1), strides=(2, 2), padding='same')(x)
    skip = layers.BatchNormalization()(skip)

    x = layers.SeparableConv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2), strides=(2, 2), padding='same')(x)
    x = layers.add([x, skip])

    x = layers.SeparableConv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(num_classes)(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate=0.0005),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    return model

model = build_xception()
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stop]
)
print("\nОцінка на тестових даних")
test_logits = model.predict(test_generator)
test_probs = tf.nn.sigmoid(test_logits).numpy().flatten()
test_preds = (test_probs > 0.5).astype(int)
test_labels = test_generator.labels

cm = confusion_matrix(test_labels, test_preds)
accuracy = accuracy_score(test_labels, test_preds)
precision, recall, fscore, _ = precision_recall_fscore_support(test_labels, test_preds, average='binary')

print("Confusion Matrix")
print(cm)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F-Score: {fscore:.4f}")

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Помилка: неможливо відкрити відео за шляхом {video_path}")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    frames_ids = []
    frame_predictions = []
    frame_numbers = []
    frame_count = 0

    def process_batch(batch_frames, batch_ids):
        batch_np = np.array(batch_frames).astype(np.float32)
        batch_np = preprocess_input(batch_np)
        predictions = model.predict(batch_np, verbose=0)

        for i in range(len(predictions)):
            score = 1 / (1 + np.exp(-predictions[i][0]))
            frame_predictions.append(score)
            frame_numbers.append(batch_ids[i])

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        resized_frame = cv2.resize(frame, (150, 150))
        frames.append(resized_frame)
        frames_ids.append(frame_count)

        if len(frames) == batch_size:
            process_batch(frames, frames_ids)
            frames = []
            frames_ids = []

        frame_count += 1

    if frames:
        process_batch(frames, frames_ids)

    cap.release()

    threshold = 0.5
    is_object_present = False
    start_time = 0
    intervals = []

    for i, score in enumerate(frame_predictions):
        current_time = frame_numbers[i] / fps

        if score >= threshold and not is_object_present:
            is_object_present = True
            start_time = current_time
        elif score < threshold and is_object_present:
            is_object_present = False
            end_time = current_time
            intervals.append((start_time, end_time))

    if is_object_present:
        end_time = frame_numbers[-1] / fps
        intervals.append((start_time, end_time))

    print("\nПеріоди часу, на яких з’явився об’єкт в кадрі:")
    if not intervals:
        print("Об'єкт не знайдено жодного разу.")
    else:
        for idx, (start, end) in enumerate(intervals):
            print(f"Поява {idx+1}: з {start:.2f}s  по {end:.2f}s")

Found 697 images belonging to 2 classes.
Found 174 images belonging to 2 classes.
Found 174 images belonging to 2 classes.
Epoch 1/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 29s 807ms/step - accuracy: 0.5122 - loss: 0.6868 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 388ms/step - accuracy: 0.5839 - loss: 0.6477 - val_accuracy: 0.5000 - val_loss: 0.6928
Epoch 3/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 326ms/step - accuracy: 0.5983 - loss: 0.6389 - val_accuracy: 0.5000 - val_loss: 0.6934
Epoch 4/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 387ms/step - accuracy: 0.6298 - loss: 0.6296 - val_accuracy: 0.5000 - val_loss: 0.6934
Epoch 5/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 335ms/step - accuracy: 0.6169 - loss: 0.6223 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 6/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 367ms/step - accuracy: 0.6126 - loss: 0.6302 - val_accuracy: 0.5000 - val_loss: 0.6954
Epoch 7/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 386ms/step - accuracy: 0.6557 - loss: 0.6172 - val_accuracy: 0.5000


Періоди часу, на яких з’явився об’єкт в кадрі:
Поява 1: з 0.00s  по 6.27s
